In [ ]:
from __future__ import print_function
%matplotlib inline
import numpy
import matplotlib.pyplot as plt

# Persamaan Hiperbolik - Bagian II

## Pelacakan Karakteristik

Cara umum untuk menyelesaikan persamaan diferensial parsial (PDE) hiperbolik secara analitik adalah dengan menggunakan metode karakteristik, tetapi hingga saat ini kita benar-benar belum mencoba menggunakan teori ini untuk membangun sebuah metode numerik.

Ketika memikirkan nilai solusi pada suatu titik $(x_j, t + \Delta t)$, kita tahu bahwa untuk $a > 0$, kita harus melihat ke belakang menuju titik $(x_j - a \Delta t, t)$, di mana nilai solusi pada titik tersebut akan menentukan (memberi informasi tentang) solusi pada titik yang kita minati.

Pendekatan ini bekerja dengan sangat baik — sampai kita mulai memikirkan tentang grid diskrit, di mana kita hanya mengetahui nilai solusi pada waktu $t$ di sekumpulan titik-titik diskrit saja.

![Characteristic Tracing nu != 1](./images/characteristic_tracing_1.png)

Jika garis karakteristik yang melewati titik $(x_j, t + \Delta t)$ juga tepat melewati salah satu titik grid pada waktu $t$, maka kita berada dalam kondisi baik (tidak perlu interpolasi tambahan).

Namun biasanya hal ini **tidak terjadi**, kecuali kita secara sengaja memilih $\Delta x$ dan $\Delta t$ sedemikian rupa sehingga kondisi tersebut terpenuhi.

Ternyata kondisi ini terjadi secara tepat ketika:

$$
\frac{a \Delta t}{\Delta x} = 1
$$

— yaitu **tepat sama dengan batas atas** dari hasil stabilitas kita sebelumnya (CFL number = 1).

![Characteristic Tracing nu == 1](./images/characteristic_tracing_2.png)

Demikian pula, jika $\nu < 1$, maka kita tahu bahwa garis karakteristik **tidak akan tepat mengenai** titik-titik grid.  
Perhatikan juga bahwa berkat batasan $|\nu| \leq 1$, kita tahu bahwa garis karakteristik **tidak dapat melewati** titik $x_{j-1}$ (atau lebih tepatnya: titik asal karakteristik selalu berada di antara $x_{j-1}$ dan $x_j$, tidak pernah ke kiri $x_{j-1}$).

Sebagai alternatif, kita bisa melakukan **interpolasi** antara kedua nilai yang “dipisahkan” oleh garis karakteristik tersebut, yaitu antara nilai di $x_{j-1}$ dan $x_j$ pada waktu $t_n$, untuk memperkirakan nilai pada titik asal karakteristik.  

Tunjukkan bahwa jika kita menggunakan **interpolasi linear** untuk melakukan hal ini, maka kita akan memperoleh **metode upwind** (skema upwind orde satu).

Untuk interpolasi linear, kita tahu bahwa titik perpotongan (titik asal karakteristik) berada pada  
$x_p = x_j - a \Delta t$.

1. Bentuk umum interpolasi linear (dua titik)

Kita punya dua titik:

Di $  x = x_{j-1}  $ → nilai $  U_{j-1}^n  $
Di $  x = x_j  $     → nilai $  U_j^n  $

Interpolan linear $  P_1(x)  $ yang melalui kedua titik ini ditulis dengan bentuk Lagrange (seperti yang diberikan):

$P_1(x) = \frac{x - x_{j-1}}{x_j - x_{j-1}} \, U_j^n + \frac{x - x_j}{x_{j-1} - x_j} \, U_{j-1}^n$

Karena $  x_j - x_{j-1} = \Delta x  $ dan $  x_{j-1} - x_j = -\Delta x  $, maka:
$$P_1(x) = \frac{x - x_{j-1}}{\Delta x} \, U_j^n + \frac{x - x_j}{-\Delta x} \, U_{j-1}^n$$
$$= \frac{x - x_{j-1}}{\Delta x} \, U_j^n - \frac{x - x_j}{\Delta x} \, U_{j-1}^n$$

2. Evaluasi di titik karakteristik mundur: $  x_p = x_j - a \Delta t  $
Kita ingin tahu nilai aproksimasi di titik asal karakteristik:

$$U_j^{n+1} = P_1(x_p) = P_1(x_j - a \Delta t)$$
Substitusi $  x = x_j - a \Delta t  $ ke dalam $  P_1(x)  $:
$$U_j^{n+1} = \frac{(x_j - a \Delta t) - x_{j-1}}{\Delta x} \, U_j^n - \frac{(x_j - a \Delta t) - x_j}{\Delta x} \, U_{j-1}^n$$

* Bagian pertama (koefisien $  U_j^n  $):
$$(x_j - a \Delta t) - x_{j-1} = x_j - x_{j-1} - a \Delta t = \Delta x - a \Delta t$$
Jadi:
$$\frac{(x_j - a \Delta t) - x_{j-1}}{\Delta x} = \frac{\Delta x - a \Delta t}{\Delta x} = 1 - \frac{a \Delta t}{\Delta x}$$

* Bagian kedua (koefisien $  U_{j-1}^n  $ — perhatikan tanda minus di depan):
$$(x_j - a \Delta t) - x_j = - a \Delta t$$
Jadi:
$$- \frac{(x_j - a \Delta t) - x_j}{\Delta x} = - \frac{-a \Delta t}{\Delta x} = \frac{a \Delta t}{\Delta x}$$

3. Gabungkan kembali:
$$U_j^{n+1} = \left(1 - \frac{a \Delta t}{\Delta x}\right) U_j^n + \frac{a \Delta t}{\Delta x} \, U_{j-1}^n$$

4. Ubah ke bentuk upwind klasik (langkah terakhir)
Sekarang kita tulis ulang dengan sedikit aljabar:
$$U_j^{n+1} = U_j^n - \frac{a \Delta t}{\Delta x} U_j^n + \frac{a \Delta t}{\Delta x} U_{j-1}^n$$
( karena $1 - \frac{a \Delta t}{\Delta x} = 1 - \nu $, lalu tambah dan kurang $  \nu U_j^n  $ secara implisit)
Kelompokkan suku yang mengandung $  U_j^n  $:

$$U_j^{n+1} = U_j^n + \frac{a \Delta t}{\Delta x} (U_{j-1}^n - U_j^n)$$
Atau lebih umum ditulis sebagai:
$$U_j^{n+1} = U_j^n - \frac{a \Delta t}{\Delta x} (U_j^n - U_{j-1}^n)$$


Dengan teknik yang serupa, kita juga dapat memperoleh **metode Beam-Warming** dengan menggunakan **interpolasi kuadratik**:

1. Bentuk interpolasi kuadratik Lagrange (tiga titik)
Kita menggunakan tiga titik grid di belakang (karena aliran ke kanan, $  a > 0  $):
$  x_{j-2}  $, $  x_{j-1}  $, $  x_j  $ dengan nilai $  U_{j-2}^n  $, $  U_{j-1}^n  $, $  U_j^n  $.
Polinomial Lagrange derajat 2 yang melalui ketiga titik ini adalah:
$$P_2(x) = \sum_{k=0}^{2} U_{j-k}^n \cdot \ell_k(x)$$
di mana basis Lagrange $  \ell_k(x)  $ didefinisikan sebagai:
$$\ell_k(x) = \prod_{\substack{m=0 \\ m \neq k}}^{2} \frac{x - x_{j-m}}{x_{j-k} - x_{j-m}}$$
Jika kita tulis secara eksplisit untuk $  k=0,1,2  $ (dengan indeks $  j, j-1, j-2  $):

* Untuk $  U_j^n  $ (titik $  x_j  $):

$$\ell_0(x) = \frac{(x - x_{j-1})(x - x_{j-2})}{(x_j - x_{j-1})(x_j - x_{j-2})}$$

* Untuk $  U_{j-1}^n  $ (titik $  x_{j-1}  $):

$$\ell_1(x) = \frac{(x - x_j)(x - x_{j-2})}{(x_{j-1} - x_j)(x_{j-1} - x_{j-2})}$$

* Untuk $  U_{j-2}^n  $ (titik $  x_{j-2}  $):

$$\ell_2(x) = \frac{(x - x_j)(x - x_{j-1})}{(x_{j-2} - x_j)(x_{j-2} - x_{j-1})}$$

2. Sederhanakan dengan jarak grid seragam ($  \Delta x  $)
Karena grid seragam:
$$x_j - x_{j-1} = \Delta x, \quad x_j - x_{j-2} = 2\Delta x$$
$$x_{j-1} - x_j = -\Delta x, \quad x_{j-1} - x_{j-2} = \Delta x$$
$$x_{j-2} - x_j = -2\Delta x, \quad x_{j-2} - x_{j-1} = -\Delta x$$

Substitusi ke masing-masing basis:

$  \ell_0(x) = \frac{(x - x_{j-1})(x - x_{j-2})}{(\Delta x)(2\Delta x)} = \frac{(x - x_{j-1})(x - x_{j-2})}{2 \Delta x^2}  $
$  \ell_1(x) = \frac{(x - x_j)(x - x_{j-2})}{(-\Delta x)(\Delta x)} = -\frac{(x - x_j)(x - x_{j-2})}{\Delta x^2}  $
$  \ell_2(x) = \frac{(x - x_j)(x - x_{j-1})}{(-2\Delta x)(-\Delta x)} = \frac{(x - x_j)(x - x_{j-1})}{2 \Delta x^2}  $

3. Evaluasi di titik asal karakteristik: $  x = x_j - a \Delta t  $
Sekarang kita hitung $  P_2(x_j - a \Delta t)  $:
$$U_j^{n+1} = P_2(x_j - a \Delta t)$$
Hitung setiap suku jarak relatif:

$  x - x_{j-1} = (x_j - a \Delta t) - x_{j-1} = \Delta x - a \Delta t  $
$  x - x_{j-2} = (x_j - a \Delta t) - x_{j-2} = 2\Delta x - a \Delta t  $
$  x - x_j     = (x_j - a \Delta t) - x_j     = -a \Delta t  $
Substitusi ke dalam ekspresi yang sudah disederhanakan:
$$
\begin{aligned}
P_2(x_j - a \Delta t) 
&= \frac{1}{2 \Delta x^2} U_j^n \, (\Delta x - a \Delta t)(2 \Delta x - a \Delta t) \\
&\quad + \left( -\frac{1}{\Delta x^2} \right) U_{j-1}^n \, (-a \Delta t)(2 \Delta x - a \Delta t) \\
&\quad + \frac{1}{2 \Delta x^2} U_{j-2}^n \, (-a \Delta t)(\Delta x - a \Delta t)
\end{aligned}
$$

4. Kembangkan setiap suku kuadrat
Sekarang kita kembangkan masing-masing produk:

* Suku $  U_j^n  $:

\begin{aligned}
(\Delta x - a \Delta t)(2 \Delta x - a \Delta t)
&= \Delta x \cdot 2 \Delta x + \Delta x \cdot (-a \Delta t) + (-a \Delta t) \cdot 2 \Delta x + (-a \Delta t) \cdot (-a \Delta t) \\
&= 2 \Delta x^2 - a \Delta t \Delta x - 2 a \Delta t \Delta x + a^2 \Delta t^2 \\
&= 2 \Delta x^2 - 3 a \Delta t \Delta x + a^2 \Delta t^2
\end{aligned}

* Suku $  U_{j-1}^n  $ (ingat ada tanda minus di depan):

$- (-a \Delta t)(2 \Delta x - a \Delta t) = a \Delta t (2 \Delta x - a \Delta t) = 2 a \Delta t \Delta x - a^2 \Delta t^2$

* Suku $  U_{j-2}^n  $:

$$(-a \Delta t)(\Delta x - a \Delta t) = -a \Delta t \Delta x + a^2 \Delta t^2$$
Masukkan kembali ke dalam kurung:
$$
U_j^{n+1} = \frac{1}{\Delta x^2} \Bigg[ 
\frac{1}{2} U_j^n (2 \Delta x^2 - 3 a \Delta t \, \Delta x + a^2 \Delta t^2)
+ U_{j-1}^n (2 a \Delta t \, \Delta x - a^2 \Delta t^2)
+ \frac{1}{2} U_{j-2}^n (-a \Delta t \, \Delta x + a^2 \Delta t^2) 
\Bigg]
$$

5. Bagikan $  \frac{1}{\Delta x^2}  $ ke setiap suku dan kelompokkan

* Koefisien $  U_j^n  $:

$$\frac{1}{\Delta x^2} \cdot \frac{1}{2} (2 \Delta x^2 - 3 a \Delta t \Delta x + a^2 \Delta t^2) = 1 - \frac{3 a \Delta t}{2 \Delta x} + \frac{a^2 \Delta t^2}{2 \Delta x^2}$$

* Koefisien $  U_{j-1}^n  $:

$$\frac{1}{\Delta x^2} (2 a \Delta t \Delta x - a^2 \Delta t^2) = \frac{2 a \Delta t}{\Delta x} - \frac{a^2 \Delta t^2}{\Delta x^2}$$

* Koefisien $  U_{j-2}^n  $:

$$\frac{1}{\Delta x^2} \cdot \frac{1}{2} (-a \Delta t \Delta x + a^2 \Delta t^2) = -\frac{a \Delta t}{2 \Delta x} + \frac{a^2 \Delta t^2}{2 \Delta x^2}$$

6. Gabungkan semua menjadi bentuk akhir (Beam-Warming)

$$
U_j^{n+1} = U_j^n 
- \frac{a \Delta t}{2 \Delta x} \bigl( 3 U_j^n - 4 U_{j-1}^n + U_{j-2}^n \bigr)
+ \frac{a^2 \Delta t^2}{2 \Delta x^2} \bigl( U_j^n - 2 U_{j-1}^n + U_{j-2}^n \bigr)
$$

Kelompokkan berdasarkan orde:

* Bagian orde 1 (upwind-like):

$$- \frac{a \Delta t}{2 \Delta x} (3 U_j^n - 4 U_{j-1}^n + U_{j-2}^n)$$

* Bagian orde 2 (dispersive correction):

$$+ \frac{a^2 \Delta t^2}{2 \Delta x^2} (U_j^n - 2 U_{j-1}^n + U_{j-2}^n)$$

Jadi akhirnya:

$U_j^{n+1} = U_j^n - \frac{a \Delta t}{2 \Delta x} (3 U_j^n - 4 U_{j-1}^n + U_{j-2}^n) + \frac{a^2 \Delta t^2}{2 \Delta x^2} (U_j^n - 2 U_{j-1}^n + U_{j-2}^n)$

## Kondisi Courant-Friedrichs-Lewy (CFL)

Salah satu hasil menarik dari analisis karakteristik kita adalah bahwa kriteria stabilitas menyebabkan titik potong garis karakteristik dengan garis waktu $t_n$ berada di dalam interval $[x_{j-1}, x_j]$ ketika $a > 0$.  

Hal ini mencerminkan prinsip yang lebih umum mengenai stabilitas metode numerik untuk persamaan diferensial parsial (PDE), yang pertama kali dikemukakan oleh Courant, Friedrichs, dan Lewy, dan biasa dikenal dengan nama **kondisi CFL**.

Kondisi stabilitas yang telah kita amati berulang kali, yaitu

$$
\nu = \left| \frac{a \Delta t}{\Delta x} \right| \leq 1
$$

ternyata merupakan **kondisi perlu** (necessary condition) bagi metode-metode yang dikembangkan untuk menyelesaikan persamaan adveksi (transport).  

Nilai $\nu$ ini sering disebut **bilangan Courant** (Courant number) justru karena alasan tersebut.